In [1]:
# Remove unwanted warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Data Management
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
from pandas_datareader.data import DataReader
from ta import add_all_ta_features

# Feature Engineering
from sklearn.preprocessing import StandardScaler

# Statistics
from statsmodels.tsa.stattools import adfuller

# Machine Learning
from sklearn.cluster import KMeans
from sklearn import metrics
from kneed import KneeLocator

# Reporting
import matplotlib.pyplot as plt

# Operating System
import os
import gc

In [2]:
# Global Variables
CSV_FILENAME = "stocks.csv"
PARQET_FILENAME = "stocks.parquet"
WORKING_DIR = "data/"
FEATURES = ["gvkey"]

### Data Extraction

In [3]:
if not os.path.exists(os.path.join(
        WORKING_DIR, CSV_FILENAME
    )):
    # read sample data
    file_path = os.path.join(
        WORKING_DIR, "ret_sample.csv"
    )
    raw = pl.read_csv(file_path)
    raw = raw.filter(pl.col("excntry").is_in(["CAN","USA"]))
    # write the North American csv
    raw.write_csv(os.path.join(WORKING_DIR, CSV_FILENAME))

if not os.path.exists(os.path.join(
    WORKING_DIR, PARQET_FILENAME
    )):
    # write the parquet file (more memory efficient)
    raw = pd.read_csv(os.path.join(WORKING_DIR, CSV_FILENAME), dtype={4: str})
    raw.to_parquet(PARQET_FILENAME, index=False, compression="snappy")

# read the parquet file
raw = pd.read_parquet(os.path.join(WORKING_DIR, PARQET_FILENAME))




In [4]:
# Display basic information
print("DataFrame Head:")
raw.head()

DataFrame Head:


,id,date,ret_eom,gvkey,iid,excntry,stock_ret,year,month,char_date,...,betadown_252d,prc_highprc_252d,corr_1260d,betabab_1260d,rmax5_rvol_21d,age,qmj,qmj_prof,qmj_growth,qmj_safety
0,comp_001081_01C,20050228,20050228,1081.0,01C,CAN,-0.143457,2005,2,20050131,...,0.779315,0.672204,0.387781,0.845865,0.805580,541,-1.508294,-0.994164,-0.832048,-1.017248
1,comp_001096_01C,20050228,20050228,1096.0,01C,CAN,0.028077,2005,2,20050131,...,0.445162,0.937664,0.245148,0.456872,0.923214,517,-0.706080,-0.247574,-0.155802,-0.485635
2,comp_001117_02,20050228,20050228,1117.0,02,USA,-0.168627,2005,2,20050131,...,1.073565,0.708333,0.124188,0.863334,0.898113,373,1.344458,1.601108,1.612067,-0.566631
3,comp_001186_01C,20050228,20050228,1186.0,01C,CAN,0.149056,2005,2,20050131,...,1.326215,0.774557,0.174888,0.399060,0.777183,385,1.123762,0.154734,1.196690,0.939661
4,comp_001243_01C,20050228,20050228,1243.0,01C,CAN,0.006239,2005,2,20050131,...,1.259809,0.764020,0.474511,0.908097,0.883391,661,-0.719340,-0.170207,-0.825418,0.077587


In [5]:
# the features that we need for K-Means Clustering
k_means_features = ["id", "gvkey", "excntry", "beta_60m", "betadown_252d", "ivol_capm_252d", "corr_1260d",
                    "dolvol_126d", "bidaskhl_21d", "market_equity", "be_me", "ebitda_mev", "ni_be"]
raw = raw[k_means_features]
raw

,id,gvkey,excntry,beta_60m,betadown_252d,ivol_capm_252d,corr_1260d,dolvol_126d,bidaskhl_21d,market_equity,be_me,ebitda_mev,ni_be
0,comp_001081_01C,1081.0,CAN,0.650452,0.779315,0.017210,0.387781,1.547222e+07,0.004671,2398.152284,1.254719,0.075173,-0.020637
1,comp_001096_01C,1096.0,CAN,0.423608,0.445162,0.015467,0.245148,1.708334e+05,0.008006,301.116426,1.512289,0.094516,-0.009665
2,comp_001117_02,1117.0,USA,1.636192,1.073565,0.049833,0.124188,5.934347e+04,0.018585,32.808300,0.329155,0.104697,0.192055
3,comp_001186_01C,1186.0,CAN,0.220041,1.326215,0.015602,0.174888,4.953975e+06,0.004598,1099.753789,0.454720,0.058404,0.069303
4,comp_001243_01C,1243.0,CAN,0.822356,1.259809,0.012803,0.474511,4.715816e+07,0.005936,14740.873131,0.806940,0.112443,0.060446
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398802,comp_330888_01,330888.0,USA,NaN,1.016282,0.025251,NaN,1.279750e+07,0.003889,2519.829400,0.874858,0.279338,0.342114
1398803,comp_343592_01,343592.0,USA,NaN,0.772162,0.074189,0.204931,6.124106e+05,0.053550,37.833420,0.480246,-0.120431,-0.235047
1398804,comp_347085_01,347085.0,USA,NaN,1.074470,0.038956,0.279023,1.599273e+06,0.015970,1714.561500,0.096810,0.060085,0.287705
1398805,comp_349705_01,349705.0,USA,NaN,1.214794,0.025960,NaN,8.284644e+05,0.004755,278.369000,0.555805,0.095691,0.136014


In [6]:
# find the mean of each feature for each company
numeric_profiles = raw.groupby("gvkey").mean(numeric_only=True)

# the country is also relevant for finding out if 2 stocks are related, so we one-hot encode the country
categorical_profiles = raw.groupby("gvkey").agg({
    "excntry": "first",
}).reset_index()
categorical_dummies = pd.get_dummies(categorical_profiles, columns=["excntry"], prefix="country")
stock_profiles = numeric_profiles.merge(categorical_dummies, on='gvkey')

In [7]:
# clean up data we don't need to save memory
del raw
gc.collect()
stock_profiles

,gvkey,beta_60m,betadown_252d,ivol_capm_252d,corr_1260d,dolvol_126d,bidaskhl_21d,market_equity,be_me,ebitda_mev,ni_be,country_CAN,country_USA
0,1004.0,1.564339,1.277598,0.021277,0.591908,9.585726e+06,0.007790,1150.967074,0.837512,0.099809,0.061662,0,1
1,1013.0,2.244785,1.352211,0.031010,0.458732,3.225269e+07,0.009877,1559.911045,0.619686,0.115948,-0.259128,0,1
2,1034.0,0.813737,0.963789,0.025029,0.350345,1.627687e+07,0.007756,996.349751,0.988099,0.143128,-0.063118,0,1
3,1045.0,1.879621,1.548444,0.029886,0.468696,3.287553e+08,0.010479,11270.776576,0.278815,0.115097,-0.069623,0,1
4,1050.0,1.281742,1.073153,0.030952,0.365469,1.425402e+06,0.012127,260.954614,0.571381,0.086780,0.020184,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14060,349854.0,NaN,NaN,NaN,NaN,NaN,0.015394,163.086929,NaN,NaN,NaN,0,1
14061,349972.0,1.290547,1.697332,0.055675,0.280319,4.340935e+06,0.021879,23.726626,1.211006,-81.443097,-0.868936,0,1
14062,349994.0,NaN,-0.292390,0.081076,NaN,2.083565e+06,0.033116,4.608393,0.763720,-11.183259,-6.260659,0,1
14063,352262.0,NaN,0.886103,0.020961,NaN,2.153430e+06,0.007451,590.484348,1.307945,0.149052,0.179286,0,1


In [8]:
# Remove any companies with NaN values
stock_profiles.dropna(inplace=True)
print("has NaN values:", stock_profiles.isnull().values.any())
stock_profiles

has NaN values: False


,gvkey,beta_60m,betadown_252d,ivol_capm_252d,corr_1260d,dolvol_126d,bidaskhl_21d,market_equity,be_me,ebitda_mev,ni_be,country_CAN,country_USA
0,1004.0,1.564339,1.277598,0.021277,0.591908,9.585726e+06,0.007790,1150.967074,0.837512,0.099809,0.061662,0,1
1,1013.0,2.244785,1.352211,0.031010,0.458732,3.225269e+07,0.009877,1559.911045,0.619686,0.115948,-0.259128,0,1
2,1034.0,0.813737,0.963789,0.025029,0.350345,1.627687e+07,0.007756,996.349751,0.988099,0.143128,-0.063118,0,1
3,1045.0,1.879621,1.548444,0.029886,0.468696,3.287553e+08,0.010479,11270.776576,0.278815,0.115097,-0.069623,0,1
4,1050.0,1.281742,1.073153,0.030952,0.365469,1.425402e+06,0.012127,260.954614,0.571381,0.086780,0.020184,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14050,345980.0,2.345066,1.948881,0.053042,0.472954,8.622944e+07,0.018402,1945.212238,1.399665,-2.163009,-0.954045,0,1
14051,347007.0,1.654614,1.930477,0.066628,0.331890,1.690989e+07,0.021031,2901.973302,0.041942,-0.131083,-0.510638,0,1
14052,347085.0,0.975676,0.797773,0.034202,0.234481,5.648356e+05,0.018264,953.735094,0.150874,0.098355,0.250665,0,1
14058,349530.0,2.529876,1.694684,0.071648,0.254467,1.954551e+06,0.033061,44.774797,3.488300,-0.999477,-0.796971,0,1


In [9]:
# Check for any infinite values
dfobj = stock_profiles.isin([np.inf, -np.inf])
count = np.isinf(dfobj).values.sum()
print("Infinite Values:", count)

# Garbage collection to save memory
del dfobj
gc.collect()

Infinite Values: 0


0

In [10]:
stock_profiles.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10381 entries, 0 to 14061
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gvkey           10381 non-null  float64
 1   beta_60m        10381 non-null  float64
 2   betadown_252d   10381 non-null  float64
 3   ivol_capm_252d  10381 non-null  float64
 4   corr_1260d      10381 non-null  float64
 5   dolvol_126d     10381 non-null  float64
 6   bidaskhl_21d    10381 non-null  float64
 7   market_equity   10381 non-null  float64
 8   be_me           10381 non-null  float64
 9   ebitda_mev      10381 non-null  float64
 10  ni_be           10381 non-null  float64
 11  country_CAN     10381 non-null  uint8  
 12  country_USA     10381 non-null  uint8  
dtypes: float64(11), uint8(2)
memory usage: 993.5 KB


### Feature Scaling

In [11]:
non_scalable = stock_profiles[["gvkey", "country_USA", "country_CAN"]].copy()
scalable = stock_profiles.drop(columns=["gvkey", "country_USA", "country_CAN"])

scaler = StandardScaler()
scaler = scaler.fit_transform(scalable)
df_scaled = pd.DataFrame(scaler, columns=scalable.columns, index=scalable.index)
df_scaled = pd.concat([df_scaled, non_scalable], axis=1)
df_scaled = df_scaled.set_index('gvkey')

del stock_profiles
gc.collect()

df_scaled

,beta_60m,betadown_252d,ivol_capm_252d,corr_1260d,dolvol_126d,bidaskhl_21d,market_equity,be_me,ebitda_mev,ni_be,country_USA,country_CAN
gvkey,,,,,,,,,,,,
1004.0,0.408025,0.548661,-0.779118,1.443917,-0.182925,-0.523534,-0.111990,-0.029958,0.039260,0.042854,1,0
1013.0,1.332481,0.698560,-0.206796,0.639104,0.099190,-0.397746,-0.091886,-0.063606,0.041005,0.029977,1,0
1034.0,-0.611745,-0.081780,-0.558511,-0.015903,-0.099647,-0.525633,-0.119590,-0.006696,0.043943,0.037845,1,0
1045.0,0.836368,1.092792,-0.272905,0.699316,3.789493,-0.361487,0.385490,-0.116261,0.040913,0.037584,1,0
1050.0,0.024089,0.137932,-0.210224,0.075494,-0.284489,-0.262135,-0.155742,-0.071068,0.037852,0.041189,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
345980.0,1.468723,1.897272,1.088731,0.725049,0.770990,0.116159,-0.072945,0.056879,-0.205345,0.002082,1,0
347007.0,0.530673,1.860298,1.887597,-0.127436,-0.091768,0.274638,-0.025912,-0.152851,0.014301,0.019881,1,0
347085.0,-0.391734,-0.415309,-0.019097,-0.716099,-0.295200,0.107858,-0.121685,-0.136024,0.039103,0.050440,1,0


### K-Means Clustering

In [12]:
# Find the optpimum number of clusters
X = df_scaled.copy()
K = range(1, 20)
distortions = []
for k in K:
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(X)
    distortions.append(kmeans.inertia_)

kl = KneeLocator(K,distortions, curve="convex", direction="decreasing")
c = kl.elbow
print("Optimum Clusters:", c)

Optimum Clusters: 8


In [13]:
# Fit K-Means Model
k_means = KMeans(n_clusters=c)
k_means.fit(X)
prediction = k_means.predict(df_scaled)